# 🍽️ S3C-Smart-Canteen: YOLO11 Segmentation Training
Notebook ini digunakan untuk melatih model deteksi sisa makanan menggunakan **YOLO11 Instance Segmentation** dengan dataset dari Roboflow dan penyimpanan otomatis ke Google Drive.

### 💡 Persiapan Penting:
1. Gunakan **Runtime GPU** (Runtime > Change runtime type > T4 GPU).
2. Siapkan **API Key** dari Roboflow Settings.

In [ ]:
# 1. Install Dependencies
!pip install ultralytics roboflow

In [ ]:
# 2. Cek GPU
import ultralytics
ultralytics.checks()

In [ ]:
# 3. Mount Google Drive
# Ini WAJIB agar model bisa disimpan otomatis ke Drive Anda
from google.colab import drive
import os

drive.mount('/content/drive')

# Tentukan folder penyimpanan di Drive
DRIVE_SAVE_PATH = '/content/drive/MyDrive/S3C_Smart_Canteen/models'
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
print(f"\n✅ Hasil training akan disimpan ke: {DRIVE_SAVE_PATH}")

In [ ]:
# 4. Download Dataset dari Roboflow
from roboflow import Roboflow

# Masukkan API Key Anda di sini
rf = Roboflow(api_key="MASUKKAN_API_KEY_ANDA_DISINI")
project = rf.workspace("NAMA_WORKSPACE").project("NAMA_PROJECT")
version = project.version(1) # Ganti nomor versi sesuai di Roboflow
dataset = version.download("yolov8") # Roboflow format yolov8 kompatibel dengan yolo11

In [ ]:
# 5. Jalankan Training YOLO11
# Model options: yolo11n-seg.pt (fast), yolo11s-seg.pt (recommended), yolo11m-seg.pt

!yolo task=segment mode=train \
    model=yolo11s-seg.pt \
    data={dataset.location}/data.yaml \
    epochs=100 \
    imgsz=640 \
    batch=16 \
    name=food_waste_seg_yolo11

In [ ]:
# 6. Validasi Model
!yolo task=segment mode=val \
    model=runs/segment/food_waste_seg_yolo11/weights/best.pt \
    data={dataset.location}/data.yaml

In [ ]:
# 7. Simpan Otomatis ke Google Drive & Download
import shutil
from datetime import datetime
from google.colab import files

best_model_local = 'runs/segment/food_waste_seg_yolo11/weights/best.pt'
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
best_model_drive = os.path.join(DRIVE_SAVE_PATH, f'food_seg_model_{timestamp}.pt')

# Copy ke Drive
shutil.copy(best_model_local, best_model_drive)
# Copy sebagai 'latest' agar mudah diambil
shutil.copy(best_model_local, os.path.join(DRIVE_SAVE_PATH, 'food_seg_model_latest.pt'))

print(f"\n✅ Model berhasil disimpan ke Drive: {best_model_drive}")
print(f"✅ Model 'latest' tersedia di: {os.path.join(DRIVE_SAVE_PATH, 'food_seg_model_latest.pt')}")

# Juga tawarkan download langsung ke browser
files.download(best_model_local)

### 🚀 Setelah Selesai:
1. Ambil file `food_seg_model_latest.pt` dari Google Drive Anda (folder `S3C_Smart_Canteen/models`).
2. Rename menjadi `food_seg_model.pt`.
3. Masukkan ke folder `models/` di proyek lokal Anda.